# Embeddings & Representation — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



11 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np
import re
from collections import Counter

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Embeddings & Representation — Student Lab

You will implement classic embedding methods offline:
- co-occurrence matrices
- PMI / PPMI
- SVD embeddings
- skip-gram with negative sampling (Word2Vec-style)

Focus: correctness, determinism, and interview-ready intuition.

## Corpus (offline)
We include repeated patterns so embeddings have signal.

In [ ]:
corpus = [
    'king is a man',
    'queen is a woman',
    'king and queen rule the kingdom',
    'man and woman are adults',
    'boy is a young man',
    'girl is a young woman',
    'paris is the capital of france',
    'berlin is the capital of germany',
    'france and germany are in europe',
    'rome is the capital of italy',
    'italy is in europe',
    'dog is an animal',
    'cat is an animal',
    'dog and cat are pets',
    'pets live with people',
]
print('n_sentences', len(corpus))


**Why this works.** # Embeddings & Representation — Instructor Lab

Solutions + short rationales. Fully offline and deterministic.

## Tokenization + vocab
Rationale: deterministic lowercase tokenization is enough for this toy lab; reserve `UNK` for robustness.

## Task 2: Implement `tokenize` + `build_vocab` (UNK at id 0).
*Section: Tokenization + vocabulary*

## Section 0 — Tokenization + vocabulary

### Task 0.1
Implement `tokenize(s)` that extracts lowercase alphabetic tokens.
Hint: regex `[a-z]+`.

### Task 0.2
Build vocab with `min_count` and reserve id 0 for `UNK`.
Return `(word2id, id2word, counts)`.

**Hints**
- regex `[a-z]+`.

In [ ]:
_WORD_RE = re.compile(r'[a-z]+')

def tokenize(s: str) -> list[str]:
    return _WORD_RE.findall(s.lower())

def build_vocab(texts: list[str], min_count: int = 1):
    cnt = Counter()
    for s in texts:
        cnt.update(tokenize(s))
    vocab = ['UNK'] + sorted([w for w, c in cnt.items() if c >= min_count])
    word2id = {w: i for i, w in enumerate(vocab)}
    id2word = {i: w for w, i in word2id.items()}
    return word2id, id2word, cnt

word2id, id2word, counts = build_vocab(corpus, min_count=1)
V = len(word2id)
check('unk0', word2id['UNK'] == 0)
print('V', V)

In [ ]:
# Checks
check('unk0', word2id['UNK'] == 0)

**Why this works.** ## Tokenization + vocab
Rationale: deterministic lowercase tokenization is enough for this toy lab; reserve `UNK` for robustness.

## Task 3: Implement `word_id` and `build_cooccurrence` with sliding window.
*Section: Co-occurrence matrix*

## Section 1 — Co-occurrence matrix

We build a target-context co-occurrence matrix `C` with a sliding window.

### Task 1.1
Implement `build_cooccurrence(texts, word2id, window=2)` returning `C` of shape (V,V).
Count context words within +/- `window` positions (excluding the center token).

FAANG gotcha: for large corpora, this should be sparse; we use dense here for simplicity.

In [ ]:
def word_id(w: str, word2id: dict[str,int]) -> int:
    return word2id.get(w, 0)

def build_cooccurrence(texts: list[str], word2id: dict[str,int], window: int = 2) -> np.ndarray:
    V = len(word2id)
    C = np.zeros((V, V), dtype=np.float64)
    for s in texts:
        ids = [word_id(w, word2id) for w in tokenize(s)]
        for i, center in enumerate(ids):
            lo = max(0, i - window)
            hi = min(len(ids), i + window + 1)
            for j in range(lo, hi):
                if j == i:
                    continue
                C[center, ids[j]] += 1.0
    return C

C = build_cooccurrence(corpus, word2id, window=2)
print('C sum', float(C.sum()), 'nnz', int((C>0).sum()))
check('shape', C.shape == (V, V))

In [ ]:
# Checks
check('shape', C.shape == (V, V))

**Why this works.** ## Co-occurrence
Rationale: co-occurrence is the classic distributional hypothesis; in production you store this sparsely.

## Task 4: Implement `ppmi(C)` returning a non-negative matrix.
*Section: PMI / PPMI*

## Section 2 — PMI / PPMI

PMI stands for **Pointwise Mutual Information**.

PMI(i,j) = log( P(i,j) / (P(i) P(j)) ).
PPMI(i,j) = max(PMI(i,j), 0).

Intuition: PMI measures how much more often two words occur together than chance would predict. If a pair appears together more than expected, PMI is positive. If it appears less than expected, PMI is negative. PPMI keeps only the positive values, which is the core idea behind PPMI.

### Task 2.1
Implement `ppmi(C)` returning a matrix same shape as `C`.
Use smoothing to avoid log(0).

In [ ]:
def ppmi(C: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    total = C.sum()
    row = C.sum(axis=1, keepdims=True)
    col = C.sum(axis=0, keepdims=True)
    Pij = C / max(total, eps)
    Pi = row / max(total, eps)
    Pj = col / max(total, eps)
    pmi = np.log((Pij + eps) / (Pi @ Pj + eps))
    return np.maximum(pmi, 0.0)

M = ppmi(C)
check('nonneg', float(M.min()) >= 0.0)

In [ ]:
# Checks
check('nonneg', float(M.min()) >= 0.0)

**Why this works.** ## PPMI
Rationale: PMI cancels out unigram frequency effects; PPMI avoids negative values that often add noise.

## Task 5: Implement `svd_embeddings(M, k)` and verify embedding shape.
*Section: SVD embeddings*

## Section 3 — SVD embeddings

We compute low-rank embeddings from PPMI.

### Task 3.1
Implement `svd_embeddings(M, k)` returning `E` of shape (V,k).
Use `np.linalg.svd` (dense).

In [ ]:
def svd_embeddings(M: np.ndarray, k: int = 8) -> np.ndarray:
    U, S, VT = np.linalg.svd(M, full_matrices=False)
    return U[:, :k] * np.sqrt(S[:k])[None, :]

E_svd = svd_embeddings(M, k=8)
check('svd_shape', E_svd.shape == (V, 8))

In [ ]:
# Checks
check('svd_shape', E_svd.shape == (V, 8))

**Why this works.** ## SVD embeddings
Rationale: low-rank factorization gives dense vectors; `U*sqrt(S)` is a common symmetric choice.

## Task 6: Task 3.2 — Similarity via cosine nearest neighbors.
*Section: SVD embeddings*

### Task 3.2 — Similarity
Implement cosine similarity and nearest neighbors for a query word.

In [ ]:
def normalize_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), eps)

def nearest_neighbors(word: str, E: np.ndarray, word2id: dict[str,int], id2word: dict[int,str], k: int = 5,
                      banned_words=None):
    E2 = normalize_rows(E)
    i = word_id(word, word2id)
    sims = E2 @ E2[i]
    idx = np.argsort(-sims)
    banned_words = set() if banned_words is None else set(banned_words)
    out = []
    for j in idx:
        w = id2word[int(j)]
        if int(j) == i or w in banned_words:
            continue
        out.append((w, float(sims[int(j)])))
        if len(out) >= k:
            break
    return out

print('neighbors_svd(king):', nearest_neighbors('king', E_svd, word2id, id2word, k=5))

In [ ]:
# Checks
_nn = nearest_neighbors('king', E_svd, word2id, id2word, k=5)
check('nn_nonempty', len(_nn) > 0)

**Why this works.** ## Similarity
Rationale: cosine similarity compares direction, not magnitude, which works better for embeddings. The optional `banned_words` filter is useful when you want to inspect content words separately from frequent scaffolding words.

## Task 7: Task 3.3 — Analogy on SVD embeddings (bridge).
*Section: SVD embeddings*

### Task 3.3 — Analogy bridge
Implement `analogy(a,b,c,E)` and test on SVD embeddings before SGNS.

In [ ]:
def analogy(a: str, b: str, c: str, E: np.ndarray, word2id: dict[str,int], id2word: dict[int,str], k: int = 5,
            banned_words=None):
    E2 = normalize_rows(E)
    ia, ib, ic = word_id(a, word2id), word_id(b, word2id), word_id(c, word2id)
    v = E2[ia] - E2[ib] + E2[ic]
    v = v / max(np.linalg.norm(v), 1e-12)
    sims = E2 @ v
    idx = np.argsort(-sims)
    out = []
    banned_words = set() if banned_words is None else set(banned_words)
    banned = {ia, ib, ic}
    for j in idx:
        w = id2word[int(j)]
        if int(j) in banned or w in banned_words:
            continue
        out.append((w, float(sims[int(j)])))
        if len(out) >= k:
            break
    return out

print('SVD king-man+woman:', analogy('king','man','woman', E_svd, word2id, id2word, k=5))

**Why this works.** ## Analogy bridge
Rationale: vector arithmetic is a useful sanity check, but not a robust evaluation metric.

## Task 8: Create training pairs `(center_id, context_id)` from a windowed corpus.
*Section: Skip-gram with Negative Sampling (SGNS)*

## Section 4 — Skip-gram with Negative Sampling (SGNS)

We train center and context embeddings so that true (center,context) pairs have high dot-product.

### Task 4.1
Create training pairs `(center_id, context_id)` from the corpus for a given window.
Return arrays `centers`, `contexts`.

In [ ]:
def make_pairs(texts: list[str], word2id: dict[str,int], window: int = 2):
    centers = []
    contexts = []
    for s in texts:
        ids = [word_id(w, word2id) for w in tokenize(s)]
        for i, c in enumerate(ids):
            lo = max(0, i-window)
            hi = min(len(ids), i+window+1)
            for j in range(lo, hi):
                if j == i:
                    continue
                centers.append(c)
                contexts.append(ids[j])
    return np.array(centers, dtype=np.int64), np.array(contexts, dtype=np.int64)

centers, contexts = make_pairs(corpus, word2id, window=2)
print('n_pairs', len(centers))
check('pairs_len', len(centers) == len(contexts))
check('pairs_nonempty', len(centers) > 0)

In [ ]:
# Checks
check('pairs_len', len(centers) == len(contexts))
check('pairs_nonempty', len(centers) > 0)

**Why this works.** ## SGNS pairs
Rationale: SGNS trains on local context pairs generated by a sliding window.

## Task 9: Implement negative sampling CDF and sampler.
*Section: Skip-gram with Negative Sampling (SGNS)*

### Task 4.2
Implement negative sampling distribution `p(w) â count(w)^{0.75}` and a sampler.
Hint: build a CDF and sample with `rng.random` + `np.searchsorted`.

**Hints**
- build a CDF and sample with `rng.random` + `np.searchsorted`.

In [ ]:
def neg_sampling_cdf(counts: Counter, word2id: dict[str,int], power: float = 0.75) -> np.ndarray:
    V = len(word2id)
    w = np.zeros((V,), dtype=np.float64)
    for word, idx in word2id.items():
        c = counts.get(word, 0)
        if idx == 0:
            c = 0
        w[idx] = float(c) ** power
    if w.sum() <= 0:
        w = np.ones_like(w)
        w[0] = 0
    p = w / w.sum()
    return np.cumsum(p)

def sample_neg(cdf: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    return np.searchsorted(cdf, rng.random(n)).astype(np.int64)

cdf = neg_sampling_cdf(counts, word2id)
negs = sample_neg(cdf, 1000, rng)
check('neg_in_range', int(negs.min()) >= 0 and int(negs.max()) < V)

In [ ]:
# Checks
check('neg_in_range', int(negs.min()) >= 0 and int(negs.max()) < V)

**Why this works.** ## Negative sampling
Rationale: `count^0.75` rebalances frequent and rare words, improving SGNS training stability.

## Task 10: Train SGNS embeddings.
*Section: Skip-gram with Negative Sampling (SGNS)*

### Task 4.3
Train SGNS embeddings.

Implement:
- `sigmoid`
- one SGD step for a (center, context) positive pair and K negatives

Hints:
- maximize `log σ(u_c · v_pos) + Σ log σ(-u_c · v_neg)`
- use gradients on dot products
- keep learning rate small

In [ ]:
# No source solution — left as an exercise.

## Task 11: Compare analogy behavior on SVD vs SGNS embeddings.
*Section: Tiny analogy checks*

## Section 5 — Tiny analogy checks

We already implemented `analogy(...)` above. Now use it to compare count-based and predictive embeddings on the same toy relation.

### Task 5.1
Run the same analogy on both `E_svd` and `E_sgns`, then inspect whether the nearest neighbors look similar or different.

In [ ]:
svd_out = analogy('king','man','woman', E_svd, word2id, id2word, k=5, banned_words=INSPECT_EXCLUDE)
sgns_out = analogy('king','man','woman', E_sgns, word2id, id2word, k=5, banned_words=INSPECT_EXCLUDE)
print('SVD king-man+woman:', svd_out)
print('SGNS king-man+woman:', sgns_out)

**Why this works.** ## Comparing SVD vs SGNS
Rationale: both methods can encode related words nearby, but they get there differently: SVD compresses a global co-occurrence matrix, while SGNS learns embeddings by predicting local context pairs.